# Dataset overview

Coverage, row counts, available timeframes and the dataset metadata produced by the Prompt #1 pipeline.

**This notebook contains no calculations.** Everything is computed by
`xauusd_quant.research` and read back from
`results/statistical_research/<timeframe>/`. Generate those first:

```
xq research-summary --timeframe 5m
xq research-summary --all
```

Nothing here is a trading signal. No transaction costs are applied anywhere.

In [ ]:
import sys
from pathlib import Path

# Work against the repository's own source tree, not an installed copy.
ROOT = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import polars as pl

from xauusd_quant import load_config
from xauusd_quant.research import load_research_config

config = load_config(ROOT / "config" / "data.yaml")
research = load_research_config(ROOT / "config" / "research.yaml")

TIMEFRAME = "5m"          # change and re-run

print(f"instrument : {config.instrument}")
print(f"timezone   : {config.timezone.describe()}")
print(f"time basis : {research.intraday.time_basis}")
print(f"results    : {research.results_path}")

In [ ]:
import json

results = research.results_dir(TIMEFRAME)
if not results.exists():
    raise SystemExit(
        f"No results for {TIMEFRAME}. Run:  xq research-summary --timeframe {TIMEFRAME}"
    )

def table(name):
    """Load a previously-generated research table."""
    path = results / f"{name}.parquet"
    return pl.read_parquet(path) if path.exists() else None

summary = json.loads((results / "summary.json").read_text(encoding="utf-8"))
print(f"generated : {summary['provenance']['generated_utc']}")
print(f"span      : {summary['bars']['first_timestamp']} -> {summary['bars']['last_timestamp']}")
print(f"bars      : {summary['bars']['count']:,}")

## Available bar data

In [ ]:
from xauusd_quant.data import DataStore

with DataStore(config) as store:
    timeframes = store.available_timeframes()
    lo, hi = store.tick_span()

print(f"tick span      : {lo} -> {hi}")
print(f"bar timeframes : {timeframes}")

rows = []
for tf in timeframes:
    files = sorted(config.bars_dir(tf).rglob("*.parquet"))
    frame = pl.scan_parquet(files).select(
        pl.len().alias("bars"),
        pl.col("timestamp").min().alias("first"),
        pl.col("timestamp").max().alias("last"),
    ).collect()
    rows.append({"timeframe": tf, **frame.row(0, named=True)})

pl.DataFrame(rows)

## Dataset metadata from the conversion pipeline

In [ ]:
meta_path = config.metadata_path / "dataset_metadata.json"
if meta_path.exists():
    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    for key in ("total_rows", "start_timestamp", "end_timestamp", "trading_days",
                "spread_mean", "spread_median", "price_min", "price_max"):
        print(f"{key:20}: {meta.get(key)}")
    print()
    print("timezone :", meta["timezone"]["description"])
    for warning in meta.get("warnings", []):
        print("WARNING  :", warning)
else:
    print("No dataset_metadata.json yet - run `xq metadata`.")

## Bar quality diagnostics

In [ ]:
diag_path = config.metadata_path / "bar_diagnostics.json"
if diag_path.exists():
    diag = json.loads(diag_path.read_text(encoding="utf-8"))
    rows = []
    for tf, report in diag["timeframes"].items():
        rows.append({
            "timeframe": tf,
            "bars": report["bars"],
            "coverage": report["coverage_ratio"],
            "missing_slots": report["missing_slots"],
            "unexplained_gaps": report["gaps_by_category"].get("holiday_or_unknown", 0),
            "mean_ticks_per_bar": report["mean_ticks_per_bar"],
        })
    display(pl.DataFrame(rows))
else:
    print("No bar_diagnostics.json yet - run `xq diagnose --all`.")